## Configuración y carga de datos

Se cargará información desde archivos de texto comprimidos, almacenados localmente para convertirlos a **RDD** _(Resilient Distributed Dataset)_.

In [1]:
from pyspark.sql import SparkSession

In [2]:
# 1. Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("TareaRDD_ProteinStats") \
    .getOrCreate()

sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/13 10:02:16 WARN Utils: Your hostname, MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.93 instead (on interface en0)
26/04/13 10:02:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 10:02:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# 2. Cargar el origen de datos como RDD
path = "../data/9606.protein.links.v12.0.txt.gz"
rdd_raw = sc.textFile(path)

print(rdd_raw.take(3))

['protein1 protein2 combined_score', '9606.ENSP00000000233 9606.ENSP00000356607 173', '9606.ENSP00000000233 9606.ENSP00000427567 154']


## Limpieza y transformación

In [4]:
# 3. Operaciones de transformación
# a. Quitar el encabezado
header = rdd_raw.first()
rdd_no_header = rdd_raw.filter(lambda line: line != header)

# b. Parsear las líneas: split por espacio y extraer la columna 3 (score)
# El formato es: protein1 protein2 combined_score
rdd_scores = rdd_no_header.map(lambda line: int(line.split(" ")[2]))

# Persistir en memoria por si se usa repetidas veces para el apartado de estadísticas más adelante
rdd_scores.cache()

PythonRDD[4] at RDD at PythonRDD.scala:58

## Estadísticas Descriptivas

In [8]:
# 4. Realizar operaciones de agregación
estatisticas = rdd_scores.stats()

print("--- Estadísticas Descriptivas del Combined Score ---")
print(f"Total de registros: {estatisticas.count()}")
print(f"Valor Promedio (Mean): {estatisticas.mean():.2f}")
print(f"Valor Máximo: {estatisticas.max()}")
print(f"Valor Mínimo: {estatisticas.min()}")
print(f"Desviación Estándar: {estatisticas.stdev():.2f}")

# Operación aritmética manual usando reduce: Suma total de todos los scores
suma_total = rdd_scores.reduce(lambda a, b: a + b)
print(f"\nSuma total de scores: {suma_total}")

--- Estadísticas Descriptivas del Combined Score ---
Total de registros: 13715404
Valor Promedio (Mean): 268.62
Valor Máximo: 999
Valor Mínimo: 150
Desviación Estándar: 156.98



Suma total de scores: 3684287294


In [9]:
spark.stop()